# 2) Adapter Training (Colab)

Headless continual adapter training with Drive-based artifact output.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import os
import random
import shutil
import sys
import time
from pathlib import Path
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

# --- Bootstrap: resolve repo root, install deps ---
try:
    from scripts.colab_repo_bootstrap import (
        install_colab_requirements,
        mount_drive_if_available,
        resolve_repo_root,
        running_in_colab,
    )
except ModuleNotFoundError:
    # Colab: auto-clone if repo is not on path
    _clone_target = Path("/content/bitirmeprojesi")
    if not (_clone_target / "scripts").exists():
        import subprocess
        _url = os.environ.get("AADS_REPO_URL", "https://github.com/EfeErim/bitirmeprojesi.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_clone_target)], check=True)
    os.chdir(_clone_target)
    sys.path.insert(0, str(_clone_target))
    from scripts.colab_repo_bootstrap import (
        install_colab_requirements,
        mount_drive_if_available,
        resolve_repo_root,
        running_in_colab,
    )

from scripts.colab_checkpointing import TrainingCheckpointManager

IN_COLAB = running_in_colab()
if IN_COLAB:
    mount_drive_if_available(force_remount=False)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

install_colab_requirements(ROOT / "colab_notebooks" / "requirements_colab.txt", IN_COLAB)

# --- Device ---
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime required. In Colab: Runtime -> Change runtime type -> GPU."
    )
DEVICE = "cuda"
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# --- Checkpoint manager ---
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_ROOT = Path(os.environ.get("AADS_DRIVE_LOG_ROOT", "/content/drive/MyDrive/aads_ulora"))
CHECKPOINT_MANAGER = TrainingCheckpointManager(DRIVE_ROOT / "telemetry" / RUN_ID, retention=3)

print(f"[SETUP] root={ROOT}  device={DEVICE}  run_id={RUN_ID}")

In [ ]:
# ── Training parameters ──────────────────────────────────────────────
# Edit these directly, then run the remaining cells top-to-bottom.

DATASET_ROOT   = "data/class_root_dataset"
CROP_NAME      = "tomato"
EPOCHS         = 25
BATCH_SIZE     = 64
LEARNING_RATE  = 3e-4
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.1
OOD_FACTOR     = 2.0
NUM_WORKERS    = 4
PREFETCH       = 2
PIN_MEMORY     = True
USE_CACHE      = False
RESUME_MODE    = "fresh"   # "fresh" or "resume"

# ─────────────────────────────────────────────────────────────────────
BASE_CONFIG = json.loads((ROOT / "config" / "base.json").read_text(encoding="utf-8"))

STATE = {
    "validated": False,
    "class_names": [],
    "runtime_dataset_root": None,
    "adapter": None,
    "loaders": None,
    "history": None,
    "calibration": None,
    "checkpoint_manager": CHECKPOINT_MANAGER,
    "resume_manifest": None,
    "best_val_loss": None,
}

# Check for existing checkpoint if resuming
if RESUME_MODE == "resume":
    try:
        STATE["resume_manifest"] = CHECKPOINT_MANAGER.get_latest()
        if STATE["resume_manifest"]:
            m = STATE["resume_manifest"]
            print(f"[RESUME] Found checkpoint: {m.get('name','?')} epoch={m.get('epoch',0)} step={m.get('global_step',0)}")
    except Exception:
        pass

print(f"[CONFIG] crop={CROP_NAME} epochs={EPOCHS} bs={BATCH_SIZE} lr={LEARNING_RATE} resume={RESUME_MODE}")

In [ ]:
raw_root = Path(DATASET_ROOT).expanduser()
if not raw_root.is_absolute():
    raw_root = (ROOT / raw_root).resolve()
if not raw_root.is_dir():
    raise RuntimeError(f"Dataset root not found: {raw_root}")

class_names = sorted(d.name for d in raw_root.iterdir() if d.is_dir())
if not class_names:
    raise RuntimeError(f"No class subdirectories in {raw_root}")

STATE["class_names"] = class_names
STATE["validated"] = True
print(f"[DATASET] root={raw_root}  classes={len(class_names)}: {class_names}")

In [ ]:
from src.adapter.independent_crop_adapter import IndependentCropAdapter
from src.utils.data_loader import create_training_loaders


def _normalize(name: str) -> str:
    n = "".join(ch.lower() if ch.isalnum() else "_" for ch in name.strip())
    while "__" in n:
        n = n.replace("__", "_")
    return n.strip("_")


def prepare_runtime_dataset_layout(class_root: Path, crop_name: str, seed: int = 42,
                                   allowed: list[str] | None = None) -> Path:
    """Split class_root into continual/val/test under data/runtime_notebook_datasets/{crop}."""
    runtime_root = ROOT / "data" / "runtime_notebook_datasets"
    crop_root = runtime_root / crop_name
    meta_path = crop_root / "_split_metadata.json"
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

    class_dirs = sorted(p for p in class_root.iterdir() if p.is_dir())
    allowed_set = set(allowed or [])
    source_key = {
        "v": 3, "root": str(class_root.resolve()), "crop": crop_name,
        "seed": seed, "dirs": [p.name for p in class_dirs],
        "allowed": sorted(allowed_set), "split": "80/10/10",
    }

    if crop_root.exists() and meta_path.exists():
        try:
            if json.loads(meta_path.read_text("utf-8")) == source_key:
                print("[DATASET] Reusing cached split layout")
                return runtime_root
        except Exception:
            pass

    if crop_root.exists():
        shutil.rmtree(crop_root)
    crop_root.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)
    used = set()
    for cls_dir in class_dirs:
        norm = _normalize(cls_dir.name)
        if not norm or (allowed_set and norm not in allowed_set) or norm in used:
            continue
        imgs = [p for p in cls_dir.rglob("*") if p.is_file() and p.suffix.lower() in image_exts]
        rng.shuffle(imgs)
        n = len(imgs)
        if n == 0:
            continue
        tr = max(1, int(0.8 * n))
        va = max(1, int(0.1 * n)) if n >= 3 else 0
        if tr + va >= n:
            va = max(0, n - tr - 1)
        splits = {"continual": imgs[:tr], "val": imgs[tr:tr+va], "test": imgs[tr+va:]}
        for split_name, files in splits.items():
            dst_dir = crop_root / split_name / norm
            dst_dir.mkdir(parents=True, exist_ok=True)
            for src in files:
                dst = dst_dir / src.name
                if not dst.exists():
                    shutil.copy2(src, dst)
        used.add(norm)

    meta_path.write_text(json.dumps(source_key, indent=2), encoding="utf-8")
    return runtime_root


if not STATE.get("validated"):
    raise RuntimeError("Run dataset validation cell first.")

crop_name = CROP_NAME.strip().lower()
class_root = Path(DATASET_ROOT).expanduser()
if not class_root.is_absolute():
    class_root = (ROOT / class_root).resolve()

# Taxonomy alignment
available = sorted({_normalize(p.name) for p in class_root.iterdir() if p.is_dir() and _normalize(p.name)})
try:
    taxonomy = json.loads((ROOT / "config" / "plant_taxonomy.json").read_text("utf-8"))
    expected = taxonomy.get("crop_specific_diseases", {}).get(crop_name, [])
    if expected:
        aligned = [n for n in sorted({"healthy", *[_normalize(d) for d in expected]}) if n in available]
    else:
        aligned = available
except Exception:
    aligned = available

if not aligned:
    raise RuntimeError(f"No usable classes for crop '{crop_name}'. Available: {available}")

STATE["class_names"] = aligned
print(f"[CLASSES] {aligned}")

runtime_data_root = prepare_runtime_dataset_layout(
    class_root=class_root, crop_name=crop_name,
    allowed=aligned,
)
STATE["runtime_dataset_root"] = runtime_data_root

# Build training config
continual_cfg = json.loads(json.dumps(BASE_CONFIG.get("training", {}).get("continual", {})))
continual_cfg["device"] = DEVICE
continual_cfg["num_epochs"] = EPOCHS
continual_cfg["batch_size"] = BATCH_SIZE
continual_cfg["learning_rate"] = LEARNING_RATE
continual_cfg["adapter"]["lora_r"] = LORA_R
continual_cfg["adapter"]["lora_alpha"] = LORA_ALPHA
continual_cfg["adapter"]["lora_dropout"] = LORA_DROPOUT
continual_cfg["ood"]["threshold_factor"] = OOD_FACTOR

adapter = IndependentCropAdapter(crop_name=crop_name, device=DEVICE)
print("[ENGINE] Initializing adapter (may download backbone)...")
adapter.initialize_engine(class_names=STATE["class_names"], config={"training": {"continual": continual_cfg}})

loaders = create_training_loaders(
    data_dir=str(runtime_data_root), crop=crop_name,
    batch_size=BATCH_SIZE, num_workers=0, use_cache=False,
)

STATE["adapter"] = adapter
STATE["loaders"] = loaders
STATE["continual_config"] = continual_cfg

trainable = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
print(f"[ENGINE] Ready. trainable_params={trainable:,}  classes={len(aligned)}")

In [ ]:
from scripts.colab_notebook_helpers import (
    build_history_snapshot,
    persist_training_curve_figure,
    persist_training_history_artifacts,
    save_notebook_checkpoint,
)

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run engine init cell first.")

adapter = STATE["adapter"]
loaders = STATE["loaders"]
checkpoint_manager = STATE["checkpoint_manager"]
val_loader = loaders.get("val")
run_id = RUN_ID
STDOUT_BATCH_INTERVAL = 50

# --- Resume ---
resume_state = None
if RESUME_MODE == "resume" and isinstance(STATE.get("resume_manifest"), dict):
    ckpt_path = str(STATE["resume_manifest"].get("path", "")).strip()
    if ckpt_path:
        try:
            resume_state = adapter.load_training_checkpoint(ckpt_path)
            STATE["resume_state"] = resume_state
            ps = resume_state.get("progress_state") or {}
            print(f"[RESUME] epoch={ps.get('epoch',0)} step={ps.get('global_step',0)}")
        except Exception as e:
            print(f"[RESUME] Failed: {e}")

existing_history = (resume_state or {}).get("history_snapshot", {})
train_loss_curve = list(existing_history.get("train_loss", []))
val_loss_curve = list(existing_history.get("val_loss", []))
val_acc_curve = list(existing_history.get("val_accuracy", []))
macro_f1_curve = list(existing_history.get("macro_f1", []))
weighted_f1_curve = list(existing_history.get("weighted_f1", []))
balanced_acc_curve = list(existing_history.get("balanced_accuracy", []))
gap_curve = list(existing_history.get("generalization_gap", []))

start_time = time.time()
last_checkpoint_step = -1
best_val_loss = float(STATE["best_val_loss"]) if STATE.get("best_val_loss") is not None else None

print(f"[TRAIN] epochs={EPOCHS} device={DEVICE} batch_interval={STDOUT_BATCH_INTERVAL}")


def _history_snapshot():
    return build_history_snapshot(
        state_history=STATE.get("history"),
        train_loss_curve=train_loss_curve, val_loss_curve=val_loss_curve,
        val_acc_curve=val_acc_curve, macro_f1_curve=macro_f1_curve,
        weighted_f1_curve=weighted_f1_curve, balanced_acc_curve=balanced_acc_curve,
        gap_curve=gap_curve,
    )


def _persist_history():
    snap = _history_snapshot()
    STATE["history"] = dict((STATE.get("history") or {}), **snap)
    persist_training_history_artifacts(root=ROOT, history_snapshot=STATE["history"], telemetry=None)
    return snap


def _checkpoint(reason, event, mark_best=False, val_loss=None):
    snap = _history_snapshot()
    record = save_notebook_checkpoint(
        checkpoint_manager=checkpoint_manager, adapter=adapter,
        reason=reason,
        event={
            "epoch": int(event.get("epoch_done", event.get("epoch", 0))),
            "batch": int(event.get("batch", 0)),
            "global_step": int(event.get("global_step", 0)),
            "elapsed_sec": float(event.get("elapsed_sec", time.time() - start_time)),
        },
        history_snapshot=snap, run_id=run_id, telemetry=None,
        mark_best=bool(mark_best),
        val_loss=(float(val_loss) if val_loss is not None else None),
    )
    if record is not None:
        STATE["resume_manifest"] = record
        print(f"[CKPT] {reason} epoch={record.get('epoch','?')} step={record.get('global_step','?')}")
    return record


def progress_cb(event):
    global last_checkpoint_step, best_val_loss

    if event.get("stop_requested"):
        return

    # --- Batch progress ---
    if "batch" in event:
        batch_num = int(event.get("batch", 0))
        if batch_num > 0 and (batch_num % STDOUT_BATCH_INTERVAL == 0):
            print(
                f"[TRAIN] epoch={event.get('epoch',0)} "
                f"batch={batch_num}/{event.get('total_batches',0)} "
                f"loss={event.get('batch_loss',0.0):.4f} "
                f"lr={event.get('lr',0.0):.6f} "
                f"speed={event.get('samples_per_sec',0.0):.1f}s/s "
                f"elapsed={event.get('elapsed_sec',0.0):.0f}s eta={event.get('eta_sec',0.0):.0f}s"
            )
        step = int(event.get("global_step", 0))
        if step > 0 and (step % 200 == 0) and step != last_checkpoint_step:
            _checkpoint("batch_200", event)
            last_checkpoint_step = step

    # --- Epoch end ---
    if "epoch_done" in event:
        train_loss_curve.append(float(event.get("epoch_loss", 0.0)))
        for key, curve in [("val_loss", val_loss_curve), ("val_accuracy", val_acc_curve),
                           ("macro_f1", macro_f1_curve), ("weighted_f1", weighted_f1_curve),
                           ("balanced_accuracy", balanced_acc_curve), ("generalization_gap", gap_curve)]:
            if key in event:
                curve.append(float(event[key]))

        # Save training curves to Drive (no inline display)
        plt.figure(figsize=(13, 3))
        plt.subplot(1, 3, 1)
        plt.plot(range(1, len(train_loss_curve)+1), train_loss_curve, marker="o", label="Train")
        if val_loss_curve:
            plt.plot(range(1, len(val_loss_curve)+1), val_loss_curve, marker="s", label="Val")
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss"); plt.grid(True, alpha=0.3); plt.legend()

        plt.subplot(1, 3, 2)
        for arr, lbl, mk in [(val_acc_curve,"Acc","^"),(macro_f1_curve,"MacroF1","d"),
                              (weighted_f1_curve,"WtdF1","x"),(balanced_acc_curve,"BalAcc","*")]:
            if arr:
                plt.plot(range(1, len(arr)+1), arr, marker=mk, label=lbl)
        plt.ylim(0, 1); plt.xlabel("Epoch"); plt.ylabel("Score"); plt.title("Metrics"); plt.grid(True, alpha=0.3); plt.legend()

        plt.subplot(1, 3, 3)
        if gap_curve:
            plt.plot(range(1, len(gap_curve)+1), gap_curve, marker="o", label="Gap")
        plt.axhline(0, color="black", lw=1, alpha=0.5)
        plt.xlabel("Epoch"); plt.ylabel("Gap"); plt.title("Gen. Gap"); plt.grid(True, alpha=0.3); plt.legend()
        plt.tight_layout()
        persist_training_curve_figure(root=ROOT, epoch_done=int(event["epoch_done"]), telemetry=None)
        plt.close("all")

        _persist_history()

        # Checkpoint
        vl = float(event["val_loss"]) if "val_loss" in event else None
        mark_best = False
        if vl is not None and (best_val_loss is None or vl < best_val_loss):
            best_val_loss = vl
            STATE["best_val_loss"] = best_val_loss
            mark_best = True
        _checkpoint("epoch_end", event, mark_best=mark_best, val_loss=vl)

        # Stdout summary
        parts = [f"[EPOCH] {event['epoch_done']}/{EPOCHS}: train_loss={event.get('epoch_loss',0.0):.4f}"]
        if "val_loss" in event: parts.append(f"val_loss={event['val_loss']:.4f}")
        if "val_accuracy" in event: parts.append(f"val_acc={event['val_accuracy']:.4f}")
        if "macro_f1" in event: parts.append(f"macro_f1={event['macro_f1']:.4f}")
        if mark_best: parts.append("★ BEST")
        print(" ".join(parts))


# --- Run training ---
train_kwargs = {"num_epochs": EPOCHS, "progress_callback": progress_cb}
if val_loader is not None:
    train_kwargs["val_loader"] = val_loader
if resume_state is not None:
    train_kwargs["resume_state"] = resume_state

try:
    train_result = adapter.train_increment(loaders["train"], **train_kwargs)
except Exception as exc:
    print(f"[TRAIN] Exception: {exc}")
    try:
        _checkpoint("exception", {
            "epoch": 0, "batch": 0,
            "global_step": int((STATE.get("history") or {}).get("global_step", 0)),
            "elapsed_sec": time.time() - start_time,
        })
    except Exception:
        pass
    raise

elapsed_total = time.time() - start_time
STATE["history"] = train_result.get("history", {})
_persist_history()
print(f"[TRAIN] Complete. elapsed={elapsed_total:.1f}s stopped_early={STATE['history'].get('stopped_early', False)}")

In [ ]:
if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run engine init cell first.")

adapter = STATE["adapter"]
val_loader = STATE["loaders"].get("val")
if val_loader is None or len(val_loader.dataset) == 0:
    raise RuntimeError("Validation loader is empty; cannot calibrate OOD.")

cal = adapter.calibrate_ood(val_loader)
STATE["calibration"] = cal

nc = cal.get("ood_calibration", {}).get("num_classes", 0)
ver = cal.get("ood_calibration", {}).get("version", 0)
print(f"[OOD] Calibration done. classes={nc} version={ver}")

In [ ]:
rt("Cell 7: adapter save started")

if STATE.get("adapter") is None:
    raise RuntimeError("Run Cell 4 first.")

checkpoint_dir = ROOT / "outputs" / "colab_notebook_training"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

STATE["adapter"].save_adapter(str(checkpoint_dir))
asset_dir = checkpoint_dir / "continual_sd_lora_adapter"

print("Saved adapter directory:", asset_dir)
if not asset_dir.exists():
    raise RuntimeError(f"Expected adapter dir missing: {asset_dir}")

for p in sorted(asset_dir.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(ROOT))

rt("Cell 7: adapter save completed")

In [ ]:
from scripts.colab_notebook_helpers import persist_validation_artifacts

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run engine init cell first.")

adapter = STATE["adapter"]
val_loader = STATE["loaders"].get("val")
if val_loader is None or len(val_loader.dataset) == 0:
    raise RuntimeError("Validation loader is empty.")

trainer = adapter._trainer
if trainer is None:
    raise RuntimeError("Trainer not initialized.")

trainer.adapter_model.eval()
trainer.classifier.eval()
trainer.fusion.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for batch in val_loader:
        images = batch["images"].to(trainer.device)
        labels = batch["labels"].to(trainer.device)
        preds = torch.argmax(trainer.forward_logits(images), dim=1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

if not y_true:
    raise RuntimeError("No validation samples.")

classes = [name for name, _ in sorted(adapter.class_to_idx.items(), key=lambda kv: kv[1])]
artifacts = persist_validation_artifacts(root=ROOT, y_true=y_true, y_pred=y_pred, classes=classes, telemetry=None)
plt.close("all")

acc = float(artifacts["report_dict"].get("accuracy", 0.0))
print(f"[VALIDATION] samples={len(y_true)} classes={len(classes)} accuracy={acc:.4f}")
print("[DONE] All artifacts saved to Drive.")